In [ ]:
from __future__ import annotations

from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import seaborn as sns
from numpy.polynomial.polynomial import Polynomial

from example_models.linear_chain import get_linear_chain_2v
from mxlpy import Model, Simulator, SurrogateProtocol, fns, npe, plot, scan, surrogates
from mxlpy.distributions import LogNormal, Normal, sample

## Neural posterior estimation


**Neural posterior estimation** answers the question: **what parameters could have generated the data I measured?**  
Here you use an ODE model and prior knowledge about the parameters of interest to create *synthetic data*.  
You then use the generated synthetic data as the *features* and the input parameters as the *targets* to train an *inverse problem*.  
Once that training is successful, the neural network can now predict the input parameters for real world data.  

<img src="assets/npe.png" style="max-height: 175px;">

You can use this technique for both steady-state as well as time course data.  
The only difference is in using `scan.time_course`.  

> Take care here to save the targets as well in case you use cached data :)

In [ ]:
# Note that now the parameters are the targets
npe_targets = sample(
    {
        "k1": LogNormal(mean=1.0, sigma=0.3),
    },
    n=1_000,
)

# And the generated data are the features
npe_features = (
    scan.steady_state(
        get_linear_chain_2v(),
        to_scan=npe_targets,
    )
    .get_args()
    .loc[:, ["y", "v2", "v3"]]
)

# It's always a good idea to check the inputs and outputs
fig, (ax1, ax2) = plot.two_axes(figsize=(6, 3), sharex=False)
_ = plot.violins(npe_features, ax=ax1)[1].set(title="Features", ylabel="Flux / a.u.")
_ = plot.violins(npe_targets, ax=ax2)[1].set(title="Targets", ylabel="Flux / a.u.")
plt.show()

### Train NPE

You can then train your neural posterior estimator using `npe.train_torch_ss_estimator` (or `npe.train_torch_time_course_estimator` if you have time course data).  


In [ ]:
estimator, losses = npe.torch.train_steady_state(
    features=npe_features,
    targets=npe_targets,
    epochs=100,
    batch_size=100,
)

ax = losses.plot(figsize=(4, 2.5))
ax.set(xlabel="epoch", ylabel="loss")
ax.set_ylim(0, None)
plt.show()

### Sanity check: do prior and posterior match?

In [ ]:
fig, (ax1, ax2) = plot.two_axes(figsize=(6, 2))

ax = sns.kdeplot(npe_targets, fill=True, ax=ax1)
ax.set_title("Prior")

posterior = estimator.predict(npe_features)
ax = sns.kdeplot(posterior, fill=True, ax=ax2)
ax.set_title("Posterior")
plt.show()

### Re-entrant training

As with the surrogates you often you don't know the amount of epochs you are going to need in order to reach the required loss.  
For the neural posterior estimation you can use the `npe.TorchSteadyStateTrainer` and `npe.TorchTimeCourseTrainer` respectively to continue training.   

In [ ]:
trainer = npe.torch.SteadyStateTrainer(
    features=npe_features,
    targets=npe_targets,
)

# Initial training
trainer.train(epochs=20, batch_size=100)
trainer.get_loss().plot(figsize=(4, 2.5)).set_ylim(0, None)
plt.show()

# Continue training
trainer.train(epochs=20, batch_size=100)
trainer.get_loss().plot(figsize=(4, 2.5)).set_ylim(0, None)
plt.show()

# Get trainer if loss is deemed suitable
estimator = trainer.get_estimator()

<div style="color: #ffffff; background-color: #04AA6D; padding: 3rem 1rem 3rem 1rem; box-sizing: border-box">
    <h2>First finish line</h2>
    With that you now know most of what you will need from a day-to-day basis about labelled models in mxlpy.
    <br />
    <br />
    Congratulations!
</div>

## Custom loss function

You can use a custom loss function by simply injecting a function that takes the predicted tensor `x` and the data `y` and produces another tensor.  

In [ ]:
from typing import TYPE_CHECKING

import torch

from mxlpy import LinearLabelMapper, Simulator
from mxlpy.distributions import sample
from mxlpy.fns import michaelis_menten_1s
from mxlpy.parallel import parallelise

if TYPE_CHECKING:
    from mxlpy import EstimatorProtocol

In [ ]:
def mean_abs(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(x - y))


trainer = npe.torch.SteadyStateTrainer(
    features=npe_features,
    targets=npe_targets,
    loss_fn=mean_abs,
)

trainer = npe.torch.TimeCourseTrainer(
    features=npe_features,
    targets=npe_targets,
    loss_fn=mean_abs,
)

## Label NPE

In [ ]:
# FIXME: todo
# Show how to change Adam settings or user other optimizers
# Show how to change the surrogate network

In [ ]:
def get_closed_cycle() -> tuple[Model, dict[str, int], dict[str, list[int]]]:
    """

    | Reaction       | Labelmap |
    | -------------- | -------- |
    | x1 ->[v1] x2   | [0, 1]   |
    | x2 ->[v2a] x3  | [0, 1]   |
    | x2 ->[v2b] x3  | [1, 0]   |
    | x3 ->[v3] x1   | [0, 1]   |

    """
    model = (
        Model()
        .add_parameters(
            {
                "vmax_1": 1.0,
                "km_1": 0.5,
                "vmax_2a": 1.0,
                "vmax_2b": 1.0,
                "km_2": 0.5,
                "vmax_3": 1.0,
                "km_3": 0.5,
            }
        )
        .add_variables({"x1": 1.0, "x2": 0.0, "x3": 0.0})
        .add_reaction(
            "v1",
            michaelis_menten_1s,
            stoichiometry={"x1": -1, "x2": 1},
            args=["x1", "vmax_1", "km_1"],
        )
        .add_reaction(
            "v2a",
            michaelis_menten_1s,
            stoichiometry={"x2": -1, "x3": 1},
            args=["x2", "vmax_2a", "km_2"],
        )
        .add_reaction(
            "v2b",
            michaelis_menten_1s,
            stoichiometry={"x2": -1, "x3": 1},
            args=["x2", "vmax_2b", "km_2"],
        )
        .add_reaction(
            "v3",
            michaelis_menten_1s,
            stoichiometry={"x3": -1, "x1": 1},
            args=["x3", "vmax_3", "km_3"],
        )
    )
    label_variables: dict[str, int] = {"x1": 2, "x2": 2, "x3": 2}
    label_maps: dict[str, list[int]] = {
        "v1": [0, 1],
        "v2a": [0, 1],
        "v2b": [1, 0],
        "v3": [0, 1],
    }
    return model, label_variables, label_maps

In [ ]:
def _worker(
    x: tuple[tuple[int, pd.Series], tuple[int, pd.Series]],
    mapper: LinearLabelMapper,
    time: float,
    initial_labels: dict[str, int | list[int]],
) -> pd.Series:
    (_, y_ss), (_, v_ss) = x
    return (
        Simulator(mapper.build_model(y_ss, v_ss, initial_labels=initial_labels))
        .simulate(time)
        .get_result()
        .unwrap_or_err()
    ).variables.iloc[-1]


def get_label_distribution_at_time(
    model: Model,
    label_variables: dict[str, int],
    label_maps: dict[str, list[int]],
    time: float,
    initial_labels: dict[str, int | list[int]],
    ss_concs: pd.DataFrame,
    ss_fluxes: pd.DataFrame,
) -> pd.DataFrame:
    mapper = LinearLabelMapper(
        model,
        label_variables=label_variables,
        label_maps=label_maps,
    )

    return pd.DataFrame(
        dict(
            parallelise(
                partial(
                    _worker, mapper=mapper, time=time, initial_labels=initial_labels
                ),
                inputs=list(
                    enumerate(
                        zip(
                            ss_concs.iterrows(),
                            ss_fluxes.iterrows(),
                            strict=True,
                        )
                    )
                ),  # type: ignore
                cache=None,
            )
        ),
        dtype=float,
    ).T


def inverse_parameter_elasticity(
    estimator: EstimatorProtocol,
    datum: pd.Series,
    *,
    normalized: bool = True,
    displacement: float = 1e-4,
) -> pd.DataFrame:
    ref = estimator.predict(datum).iloc[0, :]

    coefs = {}
    for name, value in datum.items():
        up = coefs[name] = estimator.predict(
            pd.Series(datum.to_dict() | {name: value * 1 + displacement})
        ).iloc[0, :]
        down = coefs[name] = estimator.predict(
            pd.Series(datum.to_dict() | {name: value * 1 - displacement})
        ).iloc[0, :]
        coefs[name] = (up - down) / (2 * displacement * value)

    coefs = pd.DataFrame(coefs)
    if normalized:
        coefs *= datum / ref.to_numpy()

    return coefs

In [ ]:
model, label_variables, label_maps = get_closed_cycle()

ss_concs, ss_fluxes = (
    Simulator(model)
    .update_parameters({"vmax_2a": 1.0, "vmax_2b": 0.5})
    .simulate_to_steady_state()
    .get_result()
    .unwrap_or_err()
)
mapper = LinearLabelMapper(
    model,
    label_variables=label_variables,
    label_maps=label_maps,
)

_, axs = plot.relative_label_distribution(
    mapper,
    (
        Simulator(
            mapper.build_model(
                ss_concs.iloc[-1], ss_fluxes.iloc[-1], initial_labels={"x1": 0}
            )
        )
        .simulate(5)
        .get_result()
        .unwrap_or_err()
    ).variables,
    sharey=True,
    n_cols=3,
)

axs[0, 0].set_ylabel("Relative label distribution")
axs[0, 1].set_xlabel("Time / s")
plt.show()

In [ ]:
surrogate_targets = sample(
    {
        "vmax_2b": Normal(0.5, 0.1),
    },
    n=1000,
).clip(lower=0)

ax = sns.kdeplot(surrogate_targets, fill=True)
ax.set_title("Prior")

In [ ]:
ss_concs, ss_fluxes = scan.steady_state(
    model,
    to_scan=surrogate_targets,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
_, ax = plot.violins(ss_concs, ax=ax1)
ax.set_ylabel("Concentration / a.u.")
_, ax = plot.violins(ss_fluxes, ax=ax2)
ax.set_ylabel("Flux / a.u.")

In [ ]:
surrogate_features = get_label_distribution_at_time(
    model=model,
    label_variables=label_variables,
    label_maps=label_maps,
    time=5,
    ss_concs=ss_concs,
    ss_fluxes=ss_fluxes,
    initial_labels={"x1": 0},
)
_, ax = plot.violins(surrogate_features)
ax.set_ylabel("Relative label distribution")

In [ ]:
estimator, losses = npe.torch.train_steady_state(
    features=surrogate_features,
    targets=surrogate_targets,
    batch_size=100,
    epochs=250,
)

ax = losses.plot()
ax.set_ylim(0, None)

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(8, 3),
    layout="constrained",
    sharex=True,
    sharey=False,
)

ax = sns.kdeplot(surrogate_targets, fill=True, ax=ax1)
ax.set_title("Prior")

posterior = estimator.predict(surrogate_features)

ax = sns.kdeplot(posterior, fill=True, ax=ax2)
ax.set_title("Posterior")
ax2.set_ylim(*ax1.get_ylim())
plt.show()

### Inverse parameter sensitivity

In [ ]:
_ = plot.heatmap(inverse_parameter_elasticity(estimator, surrogate_features.iloc[0]))

In [ ]:
elasticities = pd.DataFrame(
    {
        k: inverse_parameter_elasticity(estimator, i).loc["vmax_2b"]
        for k, i in surrogate_features.iterrows()
    }
).T

_ = plot.violins(elasticities)